# Spherical Sampling Background

Spherical sampling has applications on a wide range of industries. Many signals can be defined as functions on the sphere. Some examples include climate measurements and astrophysical observations. Unlike signals that are defined on a flat line or plane, this signals need to deal with the geometric constraints of a sphere.

This notebook serves to provide the background of Spherical Harmonics and Sampling that will be applied on the next notebooks.

The main reference this notebook follows is: *Fundamentals of Spherical Array Processing* by Boaz Rafaely

## The Sphere

<img src="images/spherical-coordinate-system.png" width="400">

Any function on the sphere can be represented by two angles: the **elevation** angle $\theta$, which goes from the North Pole to the South Pole, and the **azimuthal** angle $\phi$, which circles the sphere along the equator.

$\theta \in [0, \pi], \quad \phi \in [0, 2\pi]$

<img src="images/spherical-function.png" width="800">


In 1D, we can represent a signal as a weighted sum of sine and cosine waves. This is called the Fourier Series.

$$
f(\theta)
=
\sum_{n=-\infty}^{\infty}
c_n e^{in\theta}
$$

Similarly, for the sphere, the set of functions that are square-integrable on the sphere, $f(\theta, \phi) \in L_2(S^2)$, can be broken down into a weighted sum of **spherical harmonics** ($Y_n^m$). 

$$
f(\theta, \phi)
=
\sum_{n=0}^{\infty}
\sum_{m=-n}^{n}
c_{nm}Y_n^m(\theta, \phi)
$$

## The Spherical Harmonics



$$Y_n^m(\theta,\phi) = \sqrt{\frac{2n+1}{4\pi}\frac{(n-m)!}{(n+m)!}}\, P_n^m(\cos\theta)\, e^{im\phi}$$

with degree $n \in \{0, 1, 2, \dots\}$ and order $m \in \{-n, \dots, +n\}$, giving $2n+1$ harmonics at degree $n$ and $(N+1)^2$ in total up to degree $N$.

The spherical harmonic basis has three main components:

The **complex exponential term** $e^{im\phi}$ controls how the function varies around the sphere in the azimuthal direction $\phi$.

$$
e^{im\phi} = \cos(m\phi) + i\sin(m\phi).
$$

The **associated Legendre polynomial** term

$$
P_n^m(x)
=
(-1)^m(1-x^2)^{m/2}
\frac{d^m}{dx^m}P_n(x),
\qquad
x \in [-1,1].
$$

controls how the function varies in the elevation direction $\theta$.

The remaining term

$$
\sqrt{\frac{2n+1}{4\pi}\frac{(n-m)!}{(n+m)!}}
$$

is a **normalization factor**. 

Together, these components define the spherical harmonics $Y_n^m(\theta,\phi)$, forming the **complete orthonormal basis for square-integrable functions on the sphere**.

The first few spherical harmonics (Image from Fundamentals of Spherical Array Processing by Boaz Rafaely)

<img src="images/harmonics.png" width="800">

## The Spherical Fourier Transform

A function $f(\theta, \phi) \in L_2(S^2)$ can be represented using a weighted sum of spherical harmonics as

$$
f(\theta, \phi) = \sum_{n=0}^{\infty} \sum_{m=-n}^{n} c_{nm}\, Y_n^m(\theta, \phi)
$$

where $c_{nm}$ are the weights. These weights form the spherical Fourier transform of $f(\theta, \phi)$ and can be derived from $f(\theta, \phi)$ by

$$
c_{nm}
=
\int_{0}^{2\pi} \int_{0}^{\pi}
f(\theta, \phi)\, \bigl[Y_n^m(\theta, \phi)\bigr]^* \sin\theta \, d\theta \, d\phi.
$$

## Bandlimitedness

For a function that is bandlimited to a degree $N$, all harmonics above the bandlimit degree have coefficients of 0:

$$
c_n^m = 0 \quad \forall\, n > N
$$

For example, for a function bandlimited to degree 2:

<img src="images/bandlimited-pyramid.png" alt="bandlimited pyramid" width="750">

## Discrete Spherical Harmonics Transform

In the continuous case, the coefficient $c_n^m$ is computed using an integral over the entire sphere. This assumes that we know the function value $f(\theta, \phi)$ at every possible point on the sphere.

However, in practice, we can only observe the function at a finite number of sampling points:

$$
(\theta_q, \phi_q), \quad q = 1, 2, \dots, Q.
$$

This allows the continuous integral to be approximated by a finite weighted sum, with the discrete version of the SHT defined as:

$$
c_n^m \approx \sum_q w_q \, f(\theta_q, \phi_q)\,
\bigl[Y_n^m(\theta_q, \phi_q)\bigr]^*
$$

For a bandlimited function with maximum degree $N$, only coefficients up to degree $N$ need to be computed:

$$
f(\theta, \phi) = \sum_{n=0}^{N} \sum_{m=-n}^{n} c_n^m \, Y_n^m(\theta, \phi)
$$

## Numerical Computation of SH Coefficients

Structured sampling grids, such as the Driscoll-Healy and Gaussian-Legendre schemes, provide closed-form equations for the quadrature weights $w_q$, which enables the use of the discrete SHT equations. However, sample points may not always be allowed to be positioned in that structured manner, which means no closed-form weights are available.

In cases such as these, we can still compute the SH coefficients by treating the transform as a linear system. For a spherical function bandlimited to a degree $N$, the inverse SHT at sample point $(\theta_q, \phi_q)$ is:

$$
f(\theta_q, \phi_q) = \sum_{n=0}^{N} \sum_{m=-n}^{n} c_n^m \, Y_n^m(\theta_q, \phi_q), \quad q = 1, 2, \dots, Q.
$$

This is a single linear equation with $(N+1)^2$ unknowns, the coefficients $c_n^m$. Stacking the $Q$ equations provides the matrix form:

$$
\mathbf{f} = \mathbf{Y}\, \mathbf{c}
$$

where $\mathbf{f}$ holds the sample values, $\mathbf{c}$ holds the unknown coefficients, and $\mathbf{Y}$ has entries:

$$
\mathbf{Y}_{q,\,(n,m)} = Y_n^m(\theta_q, \phi_q).
$$

When $Q = (N+1)^2$, the system is critically determined and inverts directly:

$$
\mathbf{c} = \mathbf{Y}^{-1} \mathbf{f}.
$$

When $Q > (N+1)^2$, the system is overdetermined, and the least-squares solution uses the Moore–Penrose pseudoinverse:

$$
\mathbf{c} = \mathbf{Y}^{\dagger}\, \mathbf{f}, \qquad \mathbf{Y}^{\dagger} = \bigl(\mathbf{Y}^{H} \mathbf{Y}\bigr)^{-1} \mathbf{Y}^{H}.
$$

When $Q < (N+1)^2$ the system is underdetermined and no unique solution exists.

In either solvable case, the result can be written as:

$$
c_n^m = \sum_{q=1}^{Q} \alpha_{nm,q}\, f(\theta_q, \phi_q),
$$

where the $\alpha_{nm,q}$ are entries of the inverse or the pseudoinverse matrix.

To evaluate the stability of this approach, we can determine the conditioning of $\mathbf{Y}$.

Grids that provide well-distributed sample points, such as near-uniform grids, produce smaller condition numbers which lead to a more stable coefficient recovery.

-----

## Spherical Sampling Schemes

There exists established sampling schemes that determine where and how we weight each of the sample points.

For this report, we shall split them into two groups:
1. **Classical Grids** - points on elevation rings with equispaced azimuthal samples.
2. **Polyhedral Grids** - points places based on an inscribed polyhedron.

A short summary of each is written below and will be covered in more detail in the following notebooks.

### Classical Grids

Classical grids are structured in a way that places points in rings of constant latitude and longitude, which makes them efficient and simple to set-up. However, because of this, they contain more samples near the poles than the equator. Classical grids also provide closed-form quadrature weights, enabling the use of efficient algorithms.

Grids under this group include:

1. **Driscoll-Healy** - an equiangular sampling scheme at both $\theta$ and $\phi$
2. **Gauss-Legendre** - uses the roots of the Legendre Polynomial in the $\theta$ direction. and uniformly spaced in the $\phi$ direction. Uses less points than Driscoll-Healy
3. **McEwen-Wiaux** - another equiangular sampling scheme, but designed to use a more efficient sampling theorem as compared to Driscoll-Healy

From L-R: DH, GL, MW sampling schemes:

<img src="images/classical-grids.png" width="1200">

### Polyhedral Grids

Polyhedral grids sample the sphere using the geometry of a regular polyhedron incrisbed in a sphere. Sample points are located at vertices, or at projected points from the edges and subdivisions of the faces.

They are used to create a more evenly distribution sampling set of points over the sphere, which avoids the polar clustering that occurs on the classical grids. However, they do not come with closed-form quadrature weights, which requires the use of numerical solutions.

Sampling schemes based on vertices, centers of edges, and subdivided faces on a (from L-R): tetrahedron, octahedron, hexahedron, icosahedron and dodecahedron.

<img src="images/polyhedral-grids.png" width="1200">